# FER2013: ResNet18 + Compact Convolutional Transformer

Train and evaluate using the shared modules in `src/fer`.
Run this notebook from the repository root after installing
`requirements-notebook.txt` and `pip install -e .`.
The test split is reserved for the final evaluation; training uses a separate
validation folder or a stratified split of the training data.


In [ ]:
import os
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "fer").is_dir():
    raise RuntimeError("Open this notebook from the DeepLearning-HK261 repository root.")
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets

from fer.config import EMOTION_LABELS
from fer.model import CCT_ResNet
from fer.predictor import resolve_device
from fer.training import TrainConfig, build_transforms, check_classes, evaluate, train


## Configuration

Set `FER_DATA_DIR` or edit `data_dir`. Expected layout:
`train/<class>`, optional `val/<class>`, and `test/<class>`.
Class names: angry, disgust, fear, happy, neutral, sad, surprise.
Keep `num_workers=0` for Windows notebooks. Use a new `output_dir` per run.
ImageNet backbone weights may download on the first run; set `pretrained=False`
to train from scratch. No dataset or checkpoint is bundled in Git.


In [ ]:
config = TrainConfig(
    data_dir=Path(os.environ.get("FER_DATA_DIR", str(ROOT / "data" / "fer2013"))),
    output_dir=ROOT / "outputs" / "training",
    epochs=50,
    batch_size=64,
    lr=5e-4,
    backbone_lr=1e-5,
    validation_fraction=0.1,
    num_workers=0,
    seed=42,
    device="auto",
    pretrained=True,
)
print(config)
print("Device:", resolve_device(config.device))


## Training

Focal Loss with label smoothing, AdamW, cosine learning-rate decay and gradient
clipping. After validation accuracy reaches 50%, curriculum enables MixUp
and increases Random Erasing. Best validation weights and run metadata are
saved to `output_dir`. The training code never reads `test/`.


In [ ]:
result = train(config)
print("Best validation accuracy:", f"{result['best_val_accuracy']:.2%}")
print("Best checkpoint:", result["checkpoint"])


## Training history

Accuracy and macro F1 values use the range 0 to 1.


In [ ]:
history = pd.read_csv(config.output_dir / "history.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="Loss")
history.plot(x="epoch", y=["train_accuracy", "val_accuracy", "val_macro_f1"],
             ax=axes[1], title="Accuracy / macro F1")
fig.tight_layout()
plt.show()


## Final test evaluation

Reload the best validation checkpoint before evaluating on `test/`.
An older checkpoint selected using the test split cannot provide an independent
test score; retrain with this validation pipeline for a new evaluation.


In [ ]:
device = resolve_device(config.device)
best_model = CCT_ResNet().to(device)
checkpoint = config.output_dir / "cct_resnet_fer2013_best.pth"
best_model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
best_model.eval()

test_dataset = datasets.ImageFolder(config.data_dir / "test", transform=build_transforms())
check_classes(test_dataset)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False,
                         num_workers=0, pin_memory=device.type == "cuda")
test_result = evaluate(best_model, test_loader, device)
print("Test accuracy:", f"{test_result['accuracy']:.2%}")
print("Test macro F1:", f"{test_result['macro_f1']:.2%}")
print(classification_report(
    test_result["targets"], test_result["predictions"],
    labels=list(range(len(EMOTION_LABELS))), target_names=list(EMOTION_LABELS),
    digits=4, zero_division=0,
))


In [ ]:
matrix = confusion_matrix(
    test_result["targets"], test_result["predictions"],
    labels=list(range(len(EMOTION_LABELS))),
)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS)
ax.set(title="FER2013 test confusion matrix", xlabel="Predicted", ylabel="True")
fig.tight_layout()
plt.show()


## Webcam / image inference

Use the checkpoint trained above:

```powershell
.\.venv\Scripts\python.exe main.py --weights outputs/training/cct_resnet_fer2013_best.pth
.\.venv\Scripts\python.exe main.py --weights outputs/training/cct_resnet_fer2013_best.pth --image data/example.jpg --output outputs/result.jpg
```
